# «Свои вина» на vast.ai: сайт + распознавание на GPU

1. Загрузите `wine-deploy.tar.gz` в Jupyter (кнопка **Upload**) — в любую папку.
2. Выполните ячейки по порядку. Первая установка — 5–10 минут (Node, зависимости, веса SigLIP ~4.5 ГБ с Hugging Face).
3. Последняя строка ячейки запуска — публичная https-ссылка на сайт (`*.trycloudflare.com`). Она меняется при каждом перезапуске.

Распознавание слушает только `127.0.0.1:8000` на этой машине, сайт — порт 3000 (наружу через туннель).

In [ ]:
import glob, hashlib, os, subprocess, sys
EXPECTED_SIZE, EXPECTED_SHA256 = 27564034, '033c9c4fe9647f188de0ec8f22ea6aef0fa21f48cf9febc2c2a35dae28ee1272'  # filled by make_bundle.sh
ROOT = '/workspace/wine' if os.path.isdir('/workspace') else os.path.expanduser('~/wine')
found = {p for d in ('/workspace', os.path.expanduser('~'), '/tmp') for p in glob.glob(f'{d}/**/wine-deploy.tar.gz', recursive=True)}
assert found, 'загрузите wine-deploy.tar.gz через Upload'
bundle = max(found, key=os.path.getmtime)
size = os.path.getsize(bundle)
sha = hashlib.sha256(open(bundle, 'rb').read()).hexdigest()
if EXPECTED_SIZE and (size != EXPECTED_SIZE or sha != EXPECTED_SHA256):
    raise RuntimeError(f'архив загружен не полностью или не тот: {size / 1e6:.1f} из {EXPECTED_SIZE / 1e6:.1f} МБ. '
                       'Дождитесь конца загрузки (полоска внизу списка файлов) или загрузите заново, потом запустите ячейку снова')
os.makedirs(ROOT, exist_ok=True)
r = subprocess.run(['tar', '--warning=no-unknown-keyword', '-xzf', bundle, '-C', ROOT], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError(r.stderr[-1500:])
os.environ['ROOT'] = ROOT
os.environ['PY'] = sys.executable   # the scripts use the same Python (and CUDA torch) as this notebook
print('ok:', bundle, f'{size / 1e6:.1f} МБ ->', ROOT, sorted(os.listdir(ROOT)), '| python', sys.executable)

In [ ]:
# один раз: Node 22, cloudflared, Python-зависимости, веса моделей
!bash $ROOT/deploy/vast/setup.sh

In [ ]:
# запуск / перезапуск: распознавание на GPU + сайт + https-туннель (печатает ссылку)
!bash $ROOT/deploy/vast/start.sh

In [ ]:
# проверка: фото «Формула Q» через API сайта (как скрипт организаторов) и время ответа
!curl -s -w '  %{time_total}s\n' -F file=@$ROOT/deploy/vast/test.jpg http://127.0.0.1:3000/api/predict
!curl -s http://127.0.0.1:3000/api/ml/status
!cat $ROOT/run/public_url.txt 2>/dev/null

In [ ]:
# логи
!tail -n 25 $ROOT/run/ml.log; echo ----; tail -n 15 $ROOT/run/web.log; echo ----; tail -n 5 $ROOT/run/tunnel.log

In [ ]:
# остановить всё
!bash $ROOT/deploy/vast/stop.sh